# Chapter 13 &mdash; Simple TM Examples: Bit Flipper and "Contains 101"

**Concept 8 of the Chapter 13 decomposition:** *Simple TM Examples: Bit Flipper and "Contains 101"*

Two small machines that make the write-and-move discipline concrete.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter13/Concept-Simple-TM-Examples/Concept-Simple-TM-Examples.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_PDA        import *
from jove.Def_TM         import *
from jove.AnimateTM      import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


Two machines worth writing by hand.

**Bit flipper.** Sweep right, writing the complement of each cell, halt at the first
blank. It *transforms* the tape rather than deciding a language &mdash; a reminder that
TMs compute functions, not just accept.

**Contains 101.** The same state logic as the DFA of Chapter 4, but written as a TM:
the states track how much of `101` has been seen, each move writes the symbol back
unchanged and steps right, and the accepting state is a dead end.

The second machine makes the point that **every DFA is a TM** that never writes and
never moves left.

## 2. Definitions

### The two machines

In [ ]:
Flip = md2mc('''TM
!! Flip every bit, then halt in F.  F has NO outgoing transitions, which
!! is how a Jove TM signals "halt here".
I : 0 ; 1 , R -> I
I : 1 ; 0 , R -> I
I : . ; . , S -> F
''')


Has101 = md2mc('''TM
!! states track the longest prefix of 101 seen as a suffix so far
I : 0 ; 0 , R -> I
I : 1 ; 1 , R -> A       !! seen '1'
A : 1 ; 1 , R -> A       !! still just '1'
A : 0 ; 0 , R -> B       !! seen '10'
B : 1 ; 1 , R -> F       !! seen '101' -- accept (F is a dead end)
B : 0 ; 0 , R -> I       !! '100' -- start over
''')

# --- thin wrappers over Jove's TM runner --------------------------------
# run_tm(T, tape, fuel) returns (truncated-paths, haltList).  A TM HALTS
# when no transition applies, and ACCEPTS if it halts in a final state.
# So an accepting state must have NO outgoing transitions, or the machine
# will run on past it.
def tm_accepts(T, tape, fuel=200):
    trunc, halts = run_tm(T, tape if tape != '' else '.', fuel, chatty=False)
    return any(cfg[0] in T["F"] for cfg, _ in halts)

def tm_halts(T, tape, fuel=200):
    trunc, halts = run_tm(T, tape if tape != '' else '.', fuel, chatty=False)
    return len(halts) > 0

def tm_tape(T, tape, fuel=200):
    trunc, halts = run_tm(T, tape if tape != '' else '.', fuel, chatty=False)
    return [cfg[2].rstrip('.') for cfg, _ in halts]

## 3. Tests

The flipper transforms the tape.

In [ ]:
for t in ['0101', '1111', '0']:
    print("  %-7r -> %r" % (t, tm_tape(Flip, t, fuel=60)[0]))
assert tm_tape(Flip, '0101', fuel=60)[0] == '1010'
assert tm_tape(Flip, '1111', fuel=60)[0] == '0000'

Flipping twice is the identity &mdash; a cheap correctness check.

In [ ]:
from itertools import product
for k in range(1, 6):
    for p in product('01', repeat=k):
        t = ''.join(p)
        once = tm_tape(Flip, t, fuel=80)[0]
        twice = tm_tape(Flip, once, fuel=80)[0]
        assert twice == t, (t, once, twice)
print("flip(flip(t)) == t for all %d tapes up to length 5" % sum(2**k for k in range(1,6)))

`Has101` decides the language.

In [ ]:
for t in ['101', '0101', '111', '1001', '1011', '0']:
    print("  %-8r contains 101 %-6s TM accepts %s"
          % (t, '101' in t, tm_accepts(Has101, t, fuel=80)))
assert tm_accepts(Has101, '101') and not tm_accepts(Has101, '111')

Exhaustively, against the specification.

In [ ]:
bad = []
for k in range(1, 8):
    for p in product('01', repeat=k):
        t = ''.join(p)
        if tm_accepts(Has101, t, fuel=100) != ('101' in t): bad.append(t)
print("mismatches up to length 7 :", bad)
assert not bad

**Every DFA is a TM** that writes what it read and always moves right.

In [ ]:
writes_back = all(r == w for (q, r), outs in Has101["Delta"].items()
                  for (q2, w, d) in outs)
moves_right = all(d == 'R' for outs in Has101["Delta"].values()
                  for (q2, w, d) in outs)
print("writes back what it read? ", writes_back)
print("always moves right?       ", moves_right)
assert writes_back and moves_right
print("\nSo this TM is a DFA in disguise -- compare Chapter 4, Concept 9.")

## 4. Animation

`Has101`: the DFA fall-back logic, running on a tape.

*(The `display(HTML(...))` line loads the toolbar's font-awesome icons. Keep it last in the cell &mdash; it must be there for the controls to appear.)*

In [ ]:
from jove.AnimateTM import *
AnimateTM(Has101, FuseEdges=True)
display(HTML('<link rel="stylesheet" href="//stackpath.bootstrapcdn.com/font-awesome/4.7.0/css/font-awesome.min.css"/>'))

## 5. Exercises


1. Make the flipper return the head to the left end when it is done.
2. Write a TM for "even number of 1s". Does it need to write anything?
3. Which extra power does `Has101` leave unused?

In [ ]:
# Your work for the exercises above.